# MLlib: machine learning at cluster scale

_Apache Spark_

**Spark's DataFrame ML API — Transformers, Estimators, and Pipelines — runs the same training across a cluster on data that won't fit on one machine.**

MLlib is Spark's library for machine learning on data that is too big for one machine. Its modern
       interface, pyspark.ml, is built around Spark DataFrames and looks almost exactly
       like scikit-learn &mdash; the same fit/transform shapes, the same
       Pipeline idea &mdash; except the work runs across a cluster instead of on one
       processor.

       Everything is built from two kinds of objects:

---

This notebook is a practice scaffold for the **MLlib: machine learning at cluster scale** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — PySpark

### Step 1 — Start Spark and build a training/test DataFrame

We open a local `SparkSession` (one Colab VM acting as the whole cluster) and create a tiny labelled DataFrame: two numeric features `x1`, `x2` and a 0/1 label, with class 0 low-left and class 1 high-right so they separate cleanly. We split 70/30 and `cache()` the training set because iterative training scans it repeatedly.

In [ ]:
# Colab: !pip install pyspark
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Local mode: one Colab VM acts as the whole "cluster".
# On a real cluster you change ONLY .master(...) — the rest of the code is the same.
spark = (SparkSession.builder
         .master("local[*]")
         .appName("mllib-demo")
         .getOrCreate())

# A tiny Spark DataFrame: two numeric features x1,x2 and a 0/1 label.
# class 0 sits low-left, class 1 sits high-right -> cleanly separable.
rows = [
    (0.5, 0.7, 0), (0.9, 1.1, 0), (0.2, 0.4, 0), (1.1, 0.9, 0),
    (0.6, 0.3, 0), (0.8, 0.8, 0), (0.3, 1.0, 0), (1.0, 0.5, 0),
    (3.5, 3.7, 1), (3.9, 4.1, 1), (4.2, 3.4, 1), (3.1, 3.9, 1),
    (4.6, 4.3, 1), (3.8, 3.2, 1), (4.0, 4.0, 1), (3.3, 3.6, 1),
]
df = spark.createDataFrame(rows, ["x1", "x2", "label"])

train, test = df.randomSplit([0.7, 0.3], seed=42)
train.cache()   # iterative training scans the data repeatedly -> cache it

### Step 2 — Assemble the Pipeline stages

MLlib is built from **Transformers** (have only `transform`) and **Estimators** (have `fit`, which learns from data). We chain three stages: a `VectorAssembler` (Transformer) packs the feature columns into one vector; a `StandardScaler` (Estimator) learns each feature's mean and std; and `LogisticRegression` (Estimator) trains the classifier. The `Pipeline` that wraps them is itself an Estimator.

In [ ]:
# STAGE 1: Transformer — pack feature columns into ONE vector column.
assembler = VectorAssembler(inputCols=["x1", "x2"], outputCol="raw")

# STAGE 2: Estimator — fit() learns each feature's mean & std, then standardizes.
scaler = StandardScaler(inputCol="raw", outputCol="features",
                        withMean=True, withStd=True)

# STAGE 3: Estimator — fit() trains the classifier across the cluster.
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)

# Pipeline chains them; the Pipeline is itself an Estimator.
pipeline = Pipeline(stages=[assembler, scaler, lr])

### Step 3 — Fit, predict, and evaluate

Calling `fit(train)` runs every stage's `fit`/`transform` in order and returns a fitted model. `transform(test)` then adds `prediction`, `probability`, and `rawPrediction` columns. We score with `BinaryClassificationEvaluator` on area under the ROC curve — close to 1.0 here because the two classes are cleanly separable.

In [ ]:
model = pipeline.fit(train)          # runs every stage's fit/transform in order
preds = model.transform(test)        # adds prediction / probability / rawPrediction
preds.select("x1", "x2", "label", "prediction", "probability").show(truncate=False)

# Evaluate: area under the ROC (Receiver Operating Characteristic) curve.
evaluator = BinaryClassificationEvaluator(labelCol="label",
                                          rawPredictionCol="rawPrediction",
                                          metricName="areaUnderROC")
auc = evaluator.evaluate(preds)
print("areaUnderROC =", auc)         # ~1.0 here: the classes are cleanly separable

### Step 4 — Distributed hyperparameter tuning

The same Pipeline drops straight into a `CrossValidator`. We build a parameter grid over the L2 strength (`regParam`) and the ridge↔lasso mix (`elasticNetParam`), and Spark fits the pipeline on every grid point × fold across the cluster. We read off the best average metric, then stop the session to release resources.

In [ ]:
# Distributed hyperparameter tuning: same pipeline, fit on each fold.
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

grid = (ParamGridBuilder()
        .addGrid(lr.regParam, [0.0, 0.1, 0.5])          # L2 strength
        .addGrid(lr.elasticNetParam, [0.0, 1.0])        # ridge <-> lasso
        .build())
cv = CrossValidator(estimator=pipeline, estimatorParamMaps=grid,
                    evaluator=evaluator, numFolds=3, parallelism=2)
cv_model = cv.fit(train)             # 3 grid points x 2 x 3 folds, all across the cluster
print("best areaUnderROC =", max(cv_model.avgMetrics))

spark.stop()

## Visualize the data & results

_What accuracy and area-under-ROC does a standardized logistic-regression classifier reach — the same model MLlib would fit, here on a single machine for real numbers?_

### Step 1 — Load the real dataset and split it

To get real numbers we run the same model shape on the bundled breast-cancer dataset — 569 rows, 30 features, a binary target. We split it 70/30, stratified so both classes keep their proportions in each side.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# Bundled real dataset: 569 rows, 30 features, binary target.
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

### Step 2 — Standardize, then fit logistic regression

This mirrors the MLlib pipeline exactly — standardize the features, then fit logistic regression — but on a single machine via scikit-learn so the numbers are concrete. We fit the scaler on the training rows only, then apply it to both splits to avoid leaking test information.

In [ ]:
# Same shape as the MLlib pipeline: standardize, then logistic regression.
sc = StandardScaler().fit(Xtr)
Xtr_s = sc.transform(Xtr)
Xte_s = sc.transform(Xte)
clf = LogisticRegression(max_iter=1000).fit(Xtr_s, ytr)

### Step 3 — Score against a baseline

We read off accuracy and area under ROC on the held-out test set, and compare to a majority-class baseline (always predict the most common training label). The trained model lands near 0.99 on both metrics versus ~0.63 for the baseline — and this is exactly what the MLlib `VectorAssembler → StandardScaler → LogisticRegression` pipeline would produce, just distributed.

In [ ]:
proba = clf.predict_proba(Xte_s)[:, 1]
pred = clf.predict(Xte_s)
acc = accuracy_score(yte, pred)
auc = roc_auc_score(yte, proba)

# Majority-class baseline for comparison.
maj = np.bincount(ytr).argmax()
base_acc = accuracy_score(yte, np.full_like(yte, maj))

print("test rows      :", len(yte))                 # 171
print("accuracy       :", round(acc, 4))            # 0.9883
print("area under ROC :", round(auc, 4))            # 0.9981
print("baseline acc   :", round(base_acc, 4))       # 0.6257
# MLlib's VectorAssembler -> StandardScaler -> LogisticRegression, scored with
# BinaryClassificationEvaluator(areaUnderROC), gives the same numbers — distributed.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate has a 400 GB labelled event table already sitting in Spark and wants a churn classifier. They ask whether to export it and train with scikit-learn, or train in MLlib. What do you advise, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the data size relative to one machine: 400 GB will not fit in a typical box's RAM, and exporting it would mean sampling down. — _scikit-learn is single-node and in-memory, so it could only train on a sample of a 400 GB table._
- Note the data is already in Spark. — _Training in MLlib keeps everything in one engine — no export, no copy out, no second tool._
- Recommend an MLlib Pipeline: VectorAssembler → (encoders/scaler) → classifier, fit across the cluster. — _The same fit runs distributed over all 400 GB, and training on all the rows usually beats a cleverer model on a sample._

**Answer:** Train in MLlib. The table is far bigger than one machine, so scikit-learn could only see a sample, and the data is already in Spark, so an MLlib Pipeline (VectorAssembler &rarr; encoders/scaler &rarr; classifier) trains across the cluster on all 400 GB with no export. The general rule is the reverse, though: if the data fit in RAM, you would prefer scikit-learn/XGBoost &mdash; faster and richer single-node. MLlib's distributed overhead only pays off because this data does not fit.

</details>

**Problem 2.** Your MLlib code errors when you pass three numeric columns straight into LogisticRegression. What is almost certainly missing, and which kind of object (Transformer or Estimator) fixes it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that MLlib estimators read a single feature-vector column, not several separate columns. — _Passing three raw columns gives a type mismatch — the model expects one vector column named, e.g., 'features'._
- Add a VectorAssembler(inputCols=[c1,c2,c3], outputCol="features") as the first stage. — _It packs the three columns into one vector column the model can consume._
- Classify the object: VectorAssembler has only transform(), so it is a Transformer. — _It does no learning — it just maps each row's three values into one vector, deterministically._

**Answer:** You are missing a VectorAssembler &mdash; MLlib models want one column whose value is a feature vector, not several separate columns. Add VectorAssembler(inputCols=[c1,c2,c3], outputCol="features") as the first stage and point the model at featuresCol="features". VectorAssembler is a Transformer: it has only transform and learns nothing &mdash; it just packs each row's values into one vector. (Contrast an Estimator like StandardScaler or LogisticRegression, whose fit learns something from the data first.)

</details>

**Problem 3.** You wrap an MLlib Pipeline in a CrossValidator with a 4&times;3 parameter grid and 5 folds, reading the source table directly each fold. It is far slower than expected. Name two things to check.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count the fits: 4×3 grid × 5 folds = 60 full pipeline fits, each scanning the training data many times. — _Iterative training plus cross-validation reads the data over and over, so a slow read is paid repeatedly._
- Check whether the training DataFrame is cached. — _Without cache(), the cluster recomputes the DataFrame from its source on every pass — the dominant cost._
- Reconsider whether the grid/folds are larger than needed, or whether the data even needs a cluster. — _A smaller grid, fewer folds, or single-node training (if it fits in RAM) can be far cheaper._

**Answer:** First, cache the training DataFrame: train.cache() before fitting. A 4&times;3 grid over 5 folds is 60 full pipeline fits, and each iterative fit scans the data many times &mdash; without caching, the cluster recomputes the DataFrame from its source on every pass, which dominates the runtime. Second, check that the work needs a cluster at all: if the data fit in memory, the distributed overhead is wasted and a single-node grid search would be faster; and a smaller grid or fewer folds cuts the 60 fits directly.

</details>